In [3]:
# ============================================
# 检查双胞过滤前后的细胞数量变化
# ============================================

# 加载必要的包
library(Seurat)

# 定义参数（只需修改这里）🥰
prefix <- "cot"  # 修改样本名称
raw_file <- "/data/work/RNA_AraSeed/rds/cot_merge_raw.rds"   # 原始RDS完整路径
filtered_file <- "/data/work/RNA_AraSeed/rds/cot_after_QC_doublet_filtered.rds" # 过滤后RDS完整路径
outdir <- "/data/work/RNA_AraSeed/CellTypeAnno/cot"  # 输出目录完整路径

# 如果输出目录不存在则创建
if(!dir.exists(outdir)) {
  dir.create(outdir, recursive = TRUE)
  cat("✓ 创建输出目录:", outdir, "\n")
}

# ============================================
# 1. 读取两个rds文件
# ============================================
cat(paste(rep("=", 60), collapse = ""), "\n")
cat("读取RDS文件...\n")
cat(paste(rep("=", 60), collapse = ""), "\n")

# 读取原始数据（未过滤双胞）
scRNA_raw <- readRDS(raw_file)
cat("\n✓ 原始文件加载成功:", raw_file, "\n")

# 读取过滤后的数据（已做QC和双胞过滤）
scRNA_filtered <- readRDS(filtered_file)
cat("✓ 过滤后文件加载成功:", filtered_file, "\n")

# ============================================
# 2. 查看总细胞数
# ============================================
cat("\n")
cat(paste(rep("=", 60), collapse = ""), "\n")
cat("细胞数量统计\n")
cat(paste(rep("=", 60), collapse = ""), "\n")

# 原始细胞数
raw_cells <- ncol(scRNA_raw)
cat(paste0("\n【原始数据 (", prefix, "_merge_raw.rds)】"))
cat("\n  总细胞数:", raw_cells)

# 过滤后细胞数
filtered_cells <- ncol(scRNA_filtered)
cat(paste0("\n\n【过滤后数据 (", prefix, "_after_QC_doublet_filtered.rds)】"))
cat("\n  总细胞数:", filtered_cells)

# ============================================
# 3. 计算过滤掉的细胞数量和比例
# ============================================
removed_cells <- raw_cells - filtered_cells
removed_percent <- (removed_cells / raw_cells) * 100

cat("\n\n【过滤统计】")
cat("\n  移除的细胞数:", removed_cells)
cat("\n  移除比例:", sprintf("%.2f%%", removed_percent))
cat("\n  保留比例:", sprintf("%.2f%%", 100 - removed_percent))

# ============================================
# 4. 查看过滤前后的UMI分布变化
# ============================================
cat("\n\n")
cat(paste(rep("=", 60), collapse = ""), "\n")
cat("UMI分布对比（判断是否去掉了高UMI的双胞）\n")
cat(paste(rep("=", 60), collapse = ""), "\n")

# 提取UMI数据
raw_umis <- scRNA_raw$nCount_RNA
filtered_umis <- scRNA_filtered$nCount_RNA

cat("\n【原始数据 UMI统计】")
cat("\n  最小值:", min(raw_umis))
cat("\n  中位数:", median(raw_umis))
cat("\n  均值:", round(mean(raw_umis), 2))
cat("\n  最大值:", max(raw_umis))

cat("\n\n【过滤后数据 UMI统计】")
cat("\n  最小值:", min(filtered_umis))
cat("\n  中位数:", median(filtered_umis))
cat("\n  均值:", round(mean(filtered_umis), 2))
cat("\n  最大值:", max(filtered_umis))

# 检查高UMI细胞是否被移除
raw_high_umi <- sum(raw_umis > 20000)
filtered_high_umi <- sum(filtered_umis > 20000)

cat("\n\n【高UMI细胞 (>20000)】")
cat("\n  原始数据中数量:", raw_high_umi)
cat("\n  过滤后数量:", filtered_high_umi)
cat("\n  移除数量:", raw_high_umi - filtered_high_umi)

if((raw_high_umi - filtered_high_umi) > 0) {
  cat("\n  ✓ 良好: 成功移除了部分高UMI细胞（可能是双胞）")
} else {
  cat("\n  ⚠️  注意: 没有移除高UMI细胞，检查双胞过滤是否针对了高UMI群体")
}

# ============================================
# 5. 保存对比结果为文件
# ============================================
# 创建汇总表格
summary_df <- data.frame(
  Stage = c("Raw", "After_Filtering"),
  Total_Cells = c(raw_cells, filtered_cells),
  UMI_Median = c(median(raw_umis), median(filtered_umis)),
  UMI_Mean = c(round(mean(raw_umis), 2), round(mean(filtered_umis), 2)),
  UMI_Max = c(max(raw_umis), max(filtered_umis))
)

# 保存到CSV（使用 prefix）
output_file <- file.path(outdir, paste0(prefix, "_cell_filtering_summary.csv"))
write.csv(summary_df, output_file, row.names = FALSE)
cat("\n\n✓ 汇总表格已保存至:", output_file, "\n")

读取RDS文件...

✓ 原始文件加载成功: /data/work/RNA_AraSeed/rds/cot_merge_raw.rds 
✓ 过滤后文件加载成功: /data/work/RNA_AraSeed/rds/cot_after_QC_doublet_filtered.rds 

细胞数量统计

【原始数据 (cot_merge_raw.rds)】
  总细胞数: 37792

【过滤后数据 (cot_after_QC_doublet_filtered.rds)】
  总细胞数: 35332

【过滤统计】
  移除的细胞数: 2460
  移除比例: 6.51%
  保留比例: 93.49%

UMI分布对比（判断是否去掉了高UMI的双胞）

【原始数据 UMI统计】
  最小值: 1000
  中位数: 2222.5
  均值: 3877.67
  最大值: 358876

【过滤后数据 UMI统计】
  最小值: 1000
  中位数: 2149
  均值: 3389.19
  最大值: 79325

【高UMI细胞 (>20000)】
  原始数据中数量: 685
  过滤后数量: 390
  移除数量: 295
  ✓ 良好: 成功移除了部分高UMI细胞（可能是双胞）

✓ 汇总表格已保存至: /data/work/RNA_AraSeed/CellTypeAnno/cot/cot_cell_filtering_summary.csv 
